In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib

# 1. LOAD DATA
df = pd.read_csv("Index Prestasi Mahasiswa.csv")

# 2. PREPROCESSING & ENCODING
# Mengubah data kategorikal menjadi numerik agar bisa dibaca AI
df['Pemahaman_Num'] = df['Pemahaman'].replace({'Tak Paham Satupun Mk': 0, '30(%)': 30, '50(%)': 50, '100(%)': 100}).astype(int)
df['Kehadiran_Num'] = df['Kehadiran'].replace({'40(%)': 40, '60(%)': 60, '80(%)': 80, '100(%)': 100}).astype(int)

dict_datang = {'Selalu': 1, 'Jarang': 2, 'Tidak Pernah Telat': 3} # 1: Paling berisiko (selalu telat)
dict_tugas = {'Jarang': 1, 'Selalu': 3}
dict_tanya = {'Tak Pernah': 1, 'Sesekali': 2, 'Sering': 3, 'Selalu': 4}

df['Datang_Num'] = df['Datang'].map(dict_datang)
df['Tugas_Num'] = df['Kumpul_Tugas'].map(dict_tugas)
df['Tanya_Num'] = df['Bertanya'].map(dict_tanya)

# 3. LABELING KONDISI AKADEMIK (Pembuatan Target)
# Karena dataset asli tidak ada label kondisi, kita buat sistem skoring risiko awal untuk melatih AI.
def determine_condition(row):
    risk_score = 0
    if row['Kehadiran_Num'] <= 60: risk_score += 2
    if row['Pemahaman_Num'] <= 30: risk_score += 2
    if row['Tugas_Num'] == 1: risk_score += 2
    if row['Datang_Num'] == 1: risk_score += 1
    if row['IP'] < 3.3: risk_score += 2
    elif row['IP'] < 3.5: risk_score += 1

    # 0 = 🟢 MAINTAIN, 1 = 🟡 ADJUST, 2 = 🔴 SEEK SUPPORT
    if risk_score >= 3: return 2
    elif risk_score >= 1: return 1
    else: return 0

df['Target'] = df.apply(determine_condition, axis=1)

# 4. TRAINING MODEL XGBOOST
fitur = ['Datang_Num', 'Tugas_Num', 'Tanya_Num', 'Pemahaman_Num', 'Kehadiran_Num', 'IP']
X = df[fitur]
y = df['Target']

model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
model.fit(X, y)

# 5. EXPORT MODEL
joblib.dump(model, 'nilavia_xgb_model.pkl')
print("✅ Model AI NILAVIA berhasil dilatih dan disimpan sebagai 'nilavia_xgb_model.pkl'!")
print("\nDistribusi Data Latih:")
print(df['Target'].map({0: '🟢 MAINTAIN', 1: '🟡 ADJUST', 2: '🔴 SEEK SUPPORT'}).value_counts())

✅ Model AI NILAVIA berhasil dilatih dan disimpan sebagai 'nilavia_xgb_model.pkl'!

Distribusi Data Latih:
Target
🟡 ADJUST          22
🔴 SEEK SUPPORT    16
🟢 MAINTAIN        15
Name: count, dtype: int64


/tmp/ipykernel_6018/3276913464.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Pemahaman_Num'] = df['Pemahaman'].replace({'Tak Paham Satupun Mk': 0, '30(%)': 30, '50(%)': 50, '100(%)': 100}).astype(int)
/tmp/ipykernel_6018/3276913464.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Kehadiran_Num'] = df['Kehadiran'].replace({'40(%)': 40, '60(%)': 60, '80(%)': 80, '100(%)': 100}).astype(int)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [04:20:55] WARNING: /__w/xgboost/xgboost/src/lea